# Phase 255d: Classification geometry control at the 7,513-chain pool (folds 6, 7)

Phase 230 (2,055 chains, ESM2-35M): geometry beat topology alone (+0.0129 PR-AUC, 10/10 folds) and topology added a small residual
over geometry (+0.0097, 9/10 folds, PR-AUC only). Phases 240/241 found that residual grows at 650M/3B embedding scale. The untested
explanation offered for why the main pipeline keeps a residual while CryptoBench (700 chains) does not is training-set size. This
runs the identical control on the round-3 pool (3.65x the chains) at ESM2-35M.

Design identical to Phase 230: 10-fold KFold (seed 1337) over the pool (every chain is its own <30%-identity cluster, so the folds are
homology-clean), 2 seeds, 15% of each training fold for early stopping, 40 epochs, patience 8; identical 51-wide side channel for every
arm. Arms: `no-topo`, `full` (34 topology), `geom` (17), `geom+topo` (51). **Primary comparison (fixed in advance): `geom+topo` vs `geom`
on test PR-AUC, paired over the 10 folds** (this kernel runs folds [6, 7]; the five kernels are merged locally). No thresholded metric.

## Cell 1: Install

In [ ]:
!pip install -q torch-geometric==2.6.1

## Cell 2: Load the Phase 52 pool and the Phase 250 geometry (3 chunks); append 17 standardized geometry columns

In [ ]:
import json, os, time, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold
warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase52_graphs.pt"), map_location="cpu", weights_only=False)
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])
assert N_PRE == 507 and TOPO_DIM == 34, (N_PRE, TOPO_DIM)
feats = {}
for t in "abc":
    g = torch.load(locate(f"phase250{t}_geometry_scale3.pt"), map_location="cpu", weights_only=False)
    feats.update(g["feats"]); print(f"geometry chunk {t}: {g['n_chains']}/{g['n_total']} accepted")
GEOM_DIM = 17; SIDE_DIM = TOPO_DIM + GEOM_DIM

dataset, dropped = [], 0
for d in blob["dataset"]:                                       # blob order, restricted to chains with geometry (identical in every kernel)
    if (str(d.pdb_id), str(d.chain_id)) in feats: dataset.append(d)
    else: dropped += 1
del blob
assert dropped <= 0.02 * (len(dataset) + dropped), f"too many chains without geometry: {dropped}"
allg = np.concatenate([feats[(str(d.pdb_id), str(d.chain_id))] for d in dataset], 0)
g_mu = torch.from_numpy(allg.mean(0).astype(np.float32)); g_sd = torch.from_numpy(np.maximum(allg.std(0), 1e-6).astype(np.float32)); del allg
for d in dataset:
    f = feats[(str(d.pdb_id), str(d.chain_id))]; assert f.shape == (d.x.size(0), GEOM_DIM)
    d.x = torch.cat([d.x, ((torch.from_numpy(f) - g_mu) / g_sd).clamp(-5, 5)], dim=1)
del feats
print(f"chains kept {len(dataset):,}, dropped for missing geometry {dropped}; node dim {dataset[0].x.size(1)} = {N_PRE} + {TOPO_DIM} + {GEOM_DIM}")

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
if hasattr(dataset[0], "cluster"):
    assert len({d.cluster for d in dataset}) == len(dataset), "not 1 chain per cluster -- KFold would leak homologs"
    print("verified: one chain per cluster, KFold over chains is homology-clean")
else:
    print("WARNING: no `cluster` attribute on the graphs; homology-cleanliness of the folds is not re-verified here")

## Cell 3: Model and training machinery (Phase 11, unchanged)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x
class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)
class Net(nn.Module):
    def __init__(s,n_pre,td,h=128,L=4,dr=0.1):
        super().__init__(); s.n_pre=n_pre
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
    def forward(s,d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        return s.head(h).squeeze(-1)

def run_epoch(model, loader, opt=None):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    ps,ys=[],[]
    for b in loader:
        b=b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            lg=model(b); loss=crit(lg,b.y)
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        ps.append(torch.sigmoid(lg).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p,y=np.concatenate(ps),np.concatenate(ys)
    return {"pr":average_precision_score(y,p),"roc":roc_auc_score(y,p),"p":p,"y":y}

def best_mcc(p,y):
    return max(matthews_corrcoef(y,(p>=t).astype(int))
               for t in np.linspace(0.05,0.95,37) if (p>=t).sum())

## Cell 4: Arms and the nested CV loop (folds [6, 7])

In [ ]:
FOLDS_TO_RUN = [6, 7]

N_FOLDS, SEEDS, EPOCHS, PATIENCE, BATCH = 10, [0,1], 40, 8, 16
ARMS = ["no-topo", "full", "geom", "geom+topo"]
KEEP_SIDE = {"no-topo": [], "full": list(range(TOPO_DIM)), "geom": list(range(TOPO_DIM, SIDE_DIM)), "geom+topo": list(range(SIDE_DIM))}
SIDE0 = [d.x[:, N_PRE:].clone() for d in dataset]        # original side channels only; the wide ESM block is never duplicated
def apply_arm(a):
    m = torch.zeros(SIDE_DIM); m[KEEP_SIDE[a]] = 1.0
    for d, s0 in zip(dataset, SIDE0): d.x[:, N_PRE:] = s0 * m

folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=1337).split(np.arange(len(dataset))))
records = []; t0 = time.time()
for arm in ARMS:
    apply_arm(arm)
    for fi in FOLDS_TO_RUN:
        tr_idx, te_idx = folds[fi]
        for seed in SEEDS:
            rng = np.random.RandomState(500+seed)
            perm = rng.permutation(tr_idx)
            nv = max(1, int(0.15*len(perm)))
            va_ds=[dataset[i] for i in perm[:nv]]; tr_ds=[dataset[i] for i in perm[nv:]]
            te_ds=[dataset[i] for i in te_idx]
            torch.manual_seed(seed); np.random.seed(seed)
            m = Net(N_PRE, SIDE_DIM).to(DEVICE)
            opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
            tl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True); vl = DataLoader(va_ds, batch_size=BATCH)
            best, state, bad = -1, None, 0
            for ep in range(EPOCHS):
                run_epoch(m, tl, opt); v = run_epoch(m, vl); sch.step()
                if v["pr"] > best: best, state, bad = v["pr"], {k:t.clone() for k,t in m.state_dict().items()}, 0
                else:
                    bad += 1
                    if bad >= PATIENCE: break
            m.load_state_dict(state)
            te = run_epoch(m, DataLoader(te_ds, batch_size=BATCH))
            records.append({"arm": arm, "fold": fi, "seed": seed, "val_pr": float(best), "test_pr": float(te["pr"]),
                            "test_roc": float(te["roc"]), "n_test": int(len(te["y"])), "n_pos": int(te["y"].sum()), "epochs": ep + 1})
            json.dump({"records": records, "folds": FOLDS_TO_RUN}, open("/kaggle/working/phase255d_records.json", "w"), indent=2, default=float)
            print(f"  {arm:<10} fold {fi:<2} seed {seed} | val PR {best:.3f} | test PR {te['pr']:.3f} ROC {te['roc']:.3f} [{(time.time()-t0)/60:.0f}m]", flush=True)
print(f"\n{len(records)} runs in {(time.time()-t0)/60:.1f} min")